# MLP Memoization

---

## 1. Introduction

**Memoization** is an optimization technique where you **store the result of an expensive computation** so that when the same value is needed again, you simply look it up instead of recomputing it.

In the context of **Multi-Layer Perceptrons (MLPs)**, memoization refers to the practice of **saving intermediate values computed during the forward pass** so they can be directly reused during the backward pass (backpropagation) without recalculating them.

> **Core idea:** The forward pass computes values like $o_{11}$, $o_{12}$, and $\hat{y}$. The backward pass needs these *exact same values* to compute gradients. Instead of recomputing them, we **cache** (memo) them.

---

## 2. Why It Matters

Without memoization, backpropagation would need to recompute every intermediate value for every gradient. Consider a network with 100 layers — recomputing forward pass values during the backward pass would **double the computation**.

With memoization:
- Forward pass runs **once** → all intermediate values stored in memory
- Backward pass reads from memory → **no redundant computation**

This is why deep learning frameworks like PyTorch and Keras build a **computation graph** during the forward pass — it is essentially a memoization structure.

---

## 3. Our Network (Same Architecture as Before)

$$o_{11} = w^{1}_{11} x_1 + w^{1}_{21} x_2 + b_{11}$$

$$o_{12} = w^{1}_{12} x_1 + w^{1}_{22} x_2 + b_{12}$$

$$\hat{y} = w^{2}_{11} \cdot o_{11} + w^{2}_{21} \cdot o_{12} + b_{21}$$

$$L = (y - \hat{y})^2$$

The **9 parameters** that need updating: $w^{1}_{11},\ w^{1}_{21},\ w^{1}_{12},\ w^{1}_{22},\ w^{2}_{11},\ w^{2}_{21},\ b_{11},\ b_{12},\ b_{21}$

---

## 4. Forward Pass — What Gets Memoized

Take the first student: $x_1 = 80$ (IQ), $x_2 = 8$ (CGPA), $y = 3$ (LPA).

With all weights = 1, all biases = 0:

**Step 1:** Compute and **store** $o_{11}$

$$o_{11} = 1 \times 80 + 1 \times 8 + 0 = \boxed{88} \quad \longleftarrow \text{memoized}$$

**Step 2:** Compute and **store** $o_{12}$

$$o_{12} = 1 \times 80 + 1 \times 8 + 0 = \boxed{88} \quad \longleftarrow \text{memoized}$$

**Step 3:** Compute and **store** $\hat{y}$

$$\hat{y} = 1 \times 88 + 1 \times 88 + 0 = \boxed{176} \quad \longleftarrow \text{memoized}$$

**Step 4:** Compute and **store** the error signal

$$\text{error} = -2(y - \hat{y}) = -2(3 - 176) = \boxed{346} \quad \longleftarrow \text{memoized}$$

After the forward pass, our **memo cache** holds:

| Variable | Value | Used in gradient of |
|----------|-------|---------------------|
| $x_1$ | 80 | $w^{1}_{11}$, $w^{1}_{12}$ |
| $x_2$ | 8 | $w^{1}_{21}$, $w^{1}_{22}$ |
| $o_{11}$ | 88 | $w^{2}_{11}$ |
| $o_{12}$ | 88 | $w^{2}_{21}$ |
| $\hat{y}$ | 176 | error signal |
| $\text{error} = -2(y-\hat{y})$ | 346 | all 9 gradients |

---

## 5. Backward Pass — Reusing Memoized Values

Now we compute all 9 gradients. Notice how **every single one** just reads from the cache — nothing is recomputed.

### Output layer gradients

$$\frac{\partial L}{\partial w^{2}_{11}} = \underbrace{\text{error}}_{\text{memo: }346} \times \underbrace{o_{11}}_{\text{memo: }88} = 346 \times 88 = \mathbf{30{,}448}$$

$$\frac{\partial L}{\partial w^{2}_{21}} = \underbrace{\text{error}}_{\text{memo: }346} \times \underbrace{o_{12}}_{\text{memo: }88} = 346 \times 88 = \mathbf{30{,}448}$$

$$\frac{\partial L}{\partial b_{21}} = \underbrace{\text{error}}_{\text{memo: }346} \times 1 = \mathbf{346}$$

### Hidden layer gradients — neuron 1

$$\frac{\partial L}{\partial w^{1}_{11}} = \underbrace{\text{error}}_{\text{memo: }346} \times w^{2}_{11} \times \underbrace{x_1}_{\text{memo: }80} = 346 \times 1 \times 80 = \mathbf{27{,}680}$$

$$\frac{\partial L}{\partial w^{1}_{21}} = \underbrace{\text{error}}_{\text{memo: }346} \times w^{2}_{11} \times \underbrace{x_2}_{\text{memo: }8} = 346 \times 1 \times 8 = \mathbf{2{,}768}$$

$$\frac{\partial L}{\partial b_{11}} = \underbrace{\text{error}}_{\text{memo: }346} \times w^{2}_{11} = 346 \times 1 = \mathbf{346}$$

### Hidden layer gradients — neuron 2

$$\frac{\partial L}{\partial w^{1}_{12}} = \underbrace{\text{error}}_{\text{memo: }346} \times w^{2}_{21} \times \underbrace{x_1}_{\text{memo: }80} = 346 \times 1 \times 80 = \mathbf{27{,}680}$$

$$\frac{\partial L}{\partial w^{1}_{22}} = \underbrace{\text{error}}_{\text{memo: }346} \times w^{2}_{21} \times \underbrace{x_2}_{\text{memo: }8} = 346 \times 1 \times 8 = \mathbf{2{,}768}$$

$$\frac{\partial L}{\partial b_{12}} = \underbrace{\text{error}}_{\text{memo: }346} \times w^{2}_{21} = 346 \times 1 = \mathbf{346}$$

> Every gradient was computed using **only cached values** from the forward pass. Not a single intermediate value was recomputed.

---

## 6. Python Implementation — Explicit Memoization

```python
import numpy as np

# ── Dataset ───────────────────────────────────────────────────
X = np.array([[80, 8],
              [60, 9],
              [70, 5],
              [120, 7]], dtype=float)
y = np.array([3, 5, 8, 11], dtype=float)

# ── Initialize weights and biases ────────────────────────────
w1_11, w1_21 = 1.0, 1.0
w1_12, w1_22 = 1.0, 1.0
b11,   b12   = 0.0, 0.0
w2_11, w2_21 = 1.0, 1.0
b21          = 0.0

learning_rate = 0.0001
epochs        = 100

for epoch in range(epochs):
    total_loss = 0

    for j in np.random.permutation(len(X)):
        x1, x2  = X[j]
        y_true  = y[j]

        # ── Forward pass — compute AND STORE (memoize) ────────
        memo = {}                             # our cache
        memo['x1']    = x1
        memo['x2']    = x2
        memo['o11']   = w1_11 * x1 + w1_21 * x2 + b11
        memo['o12']   = w1_12 * x1 + w1_22 * x2 + b12
        memo['y_hat'] = w2_11 * memo['o11'] + w2_21 * memo['o12'] + b21

        # ── Loss ──────────────────────────────────────────────
        loss = (y_true - memo['y_hat']) ** 2
        total_loss += loss

        # ── Memoize the error signal ──────────────────────────
        memo['error'] = -2 * (y_true - memo['y_hat'])

        # ── Backward pass — READ from memo, no recomputation ──

        # Output layer
        dL_dw2_11 = memo['error'] * memo['o11']    # uses memoized o11
        dL_dw2_21 = memo['error'] * memo['o12']    # uses memoized o12
        dL_db21   = memo['error']

        # Hidden layer — neuron 1
        dL_dw1_11 = memo['error'] * w2_11 * memo['x1']  # uses memoized x1
        dL_dw1_21 = memo['error'] * w2_11 * memo['x2']  # uses memoized x2
        dL_db11   = memo['error'] * w2_11

        # Hidden layer — neuron 2
        dL_dw1_12 = memo['error'] * w2_21 * memo['x1']  # uses memoized x1
        dL_dw1_22 = memo['error'] * w2_21 * memo['x2']  # uses memoized x2
        dL_db12   = memo['error'] * w2_21

        # ── Gradient descent update ───────────────────────────
        w2_11 -= learning_rate * dL_dw2_11
        w2_21 -= learning_rate * dL_dw2_21
        b21   -= learning_rate * dL_db21

        w1_11 -= learning_rate * dL_dw1_11
        w1_21 -= learning_rate * dL_dw1_21
        b11   -= learning_rate * dL_db11

        w1_12 -= learning_rate * dL_dw1_12
        w1_22 -= learning_rate * dL_dw1_22
        b12   -= learning_rate * dL_db12

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | Avg Loss: {total_loss/len(X):.4f}")
```

---

## 7. Without vs. With Memoization

To make the contrast concrete — here is what computing $\dfrac{\partial L}{\partial w^{1}_{11}}$ would look like **without** memoization:

```python
# WITHOUT memoization — recomputes everything from scratch
def grad_w1_11_no_memo(x1, x2, y_true, w1_11, w1_21, w1_12, w1_22,
                        w2_11, w2_21, b11, b12, b21):
    o11   = w1_11 * x1 + w1_21 * x2 + b11    # recomputed
    o12   = w1_12 * x1 + w1_22 * x2 + b12    # recomputed
    y_hat = w2_11 * o11 + w2_21 * o12 + b21  # recomputed
    error = -2 * (y_true - y_hat)             # recomputed
    return error * w2_11 * x1

# WITH memoization — reads from cache
def grad_w1_11_with_memo(memo, w2_11):
    return memo['error'] * w2_11 * memo['x1']  # just a lookup
```

Calling the no-memo version for all 9 gradients means recomputing $o_{11}$, $o_{12}$, $\hat{y}$, and error **9 separate times**. With memoization, they are computed **once** and reused 9 times.

In a network with $n$ layers and $p$ parameters per layer, the savings scale as $O(n \times p)$ — this becomes critical for networks with millions of parameters.

---

## 8. Key Takeaways

- **Memoization in MLPs** means saving intermediate forward pass values ($o_{11}$, $o_{12}$, $\hat{y}$, error) so the backward pass can reuse them directly.
- Every one of the **9 gradients** depends on at least one memoized value — without caching, each would require a full forward pass recomputation.
- The **error signal** $-2(y - \hat{y})$ is the most reused memoized value — it appears in all 9 gradient formulas.
- Deep learning frameworks like **PyTorch** (`autograd`) and **Keras** implement this automatically via a **computation graph** built during the forward pass.
- Memoization is what makes backpropagation **efficient** — not just mathematically correct, but computationally practical at scale.